# Usages

## Overview

- This page provides a general pipeline for analyzing Lipidomics (LiP) data using the `lipana` package.
- The example data used here is a truncated search report from DIA-NN. This dataset includes three experiment conditions, each with three replicates, and only 1000 proteins are retained.
- The example files are located in the `"path_to/LiPAna/example_data"` directory.

In [1]:
import gzip
import shutil
from pathlib import Path

workspace = Path(".").resolve().parents[1].joinpath("example_data")
print("current workspace:", str(workspace))
# Unzip gzipped files
for file in workspace.glob("*.gz"):
    with gzip.open(file, "rb") as f_in:
        with open(file.with_suffix(""), "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)


current workspace: /home/data/LiPAna/example_data


In [2]:
import sys

sys.path.append(str(workspace.parent))
import lipana

In [3]:
from time import sleep

import polars as pl

## Define the Experiment Layout

To annotate the expected report, it is essential to establish an experiment layout. Two methods for accomplishing this are introduced below:

1. Loading from a Text File: You can load a text file containing the experiment layout. An example of such a file might look like this ("replicate" column can be omitted):

|  run | condition | replicate |
| ---: | --------: | --------: |
| run1 |      ctrl |         1 |
| run2 |      ctrl |         2 |
| run3 |      ctrl |         3 |
| run4 |      exp1 |         1 |
| run5 |      exp1 |         2 |
| run6 |      exp1 |         3 |
| run7 |      exp2 |         1 |
| run8 |      exp2 |         2 |

In [4]:
import lipana

exp_layout = lipana.ExperimentLayout.from_file(workspace.joinpath("experiment_layout.txt"))
print("The input file is:")
exp_layout.exp_df

The input file is:


run,condition,replicate
str,str,i64
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",1
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",2
"""20240404_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",3
"""20240331_YSJ-HY_5_100-400ng_A_…","""H5_Y100""",1
"""20240331_YSJ-HY_5_100-400ng_B_…","""H5_Y100""",2
"""20240404_YSJ-HY_5_100-400ng_C_…","""H5_Y100""",3
"""20240331_YSJ-HY_25_100-400ng_A…","""H25_Y100""",1
"""20240331_YSJ-HY_25_100-400ng_B…","""H25_Y100""",2
"""20240404_YSJ-HY_25_100-400ng_C…","""H25_Y100""",3


2. Initializing with a Run-to-Condition Map: Alternatively, you can define the experiment layout by passing a dictionary mapping runs to their respective conditions. This approach is useful if you need to programmatically create or adjust the layout based on specific experiment parameters without relying on an external file

In [5]:
import lipana

exp_layout = lipana.ExperimentLayout.from_run_to_condition_map(
    {
        "20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154": "H2d5_Y100",
        "20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155": "H2d5_Y100",
        "20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206": "H2d5_Y100",
        "20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157": "H5_Y100",
        "20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158": "H5_Y100",
        "20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210": "H5_Y100",
        "20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163": "H25_Y100",
        "20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164": "H25_Y100",
        "20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220": "H25_Y100",
    }
)

print("This results in the same experiment layout as the previous example, because replicate is incremented from 1")
exp_layout.exp_df

This results in the same experiment layout as the previous example, because replicate is incremented from 1


run,condition,replicate
str,str,i64
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",1
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",2
"""20240404_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",3
"""20240331_YSJ-HY_5_100-400ng_A_…","""H5_Y100""",1
"""20240331_YSJ-HY_5_100-400ng_B_…","""H5_Y100""",2
"""20240404_YSJ-HY_5_100-400ng_C_…","""H5_Y100""",3
"""20240331_YSJ-HY_25_100-400ng_A…","""H25_Y100""",1
"""20240331_YSJ-HY_25_100-400ng_B…","""H25_Y100""",2
"""20240404_YSJ-HY_25_100-400ng_C…","""H25_Y100""",3


## Load a search report

- This step involves loading and annotating the search report
  - A DIA-NN report can originate from either running DIA-NN standalone or using FragPipe, which integrates DIA-NN for DIA data processing
  - For Spectronaut reports, certain columns are required, and a template for report settings can be generated using: `lipana.report.export_sn_report_setting("path_to_store_the_setting_file.rs")`

### First Load FASTA File(s)

- Before annotating the report, it is essential to load and parse FASTA files for species and sequence information
- `lipana.parse_fasta` will read one or more FASTA files and return a `ParsedFasta` object
  - `fasta_path` can be a path points to a FASTA or a list of paths (`fasta_path=some_path` or `fasta_path=[path1, path2]`)
  - `contam_fasta_path`: this parameter will be useful to strictly exclude potential contaminants. The species of any protein entries in the given file will be annotated as "Contam" instead of its original species, including those presented in `fasta_path`
  - `contaminations`: a list of additional protein entries can be given here, and this parameter can also work alone if `contam_fasta_path` is `None`; this parameter can also be used to define all the entries that from a certain species as contaminants, by providing the species name(s) in the list
- For title parsing
  - Parameter `fasta_title_regex` is `None` by default, and the title will be parsed as the UniProt format ">sp|protein|name_species ...", which will extract "protein" and "species"
  - To set a customized regex for parsing titles, this parameter can be `str` or a list of `str`
  - For example
    - `fasta_title_regex=None` -> UniProt title
    - `fasta_title_regex=r">[^|\s]+?\|(?P<protein>[^|\s]+?)\|[^\s]+?_(?P<species>[^\s]+)[$\s].*"` -> UniProt title but by regex
    - `fasta_title_regex=r">(?P<protein>[^\ ]+).+?Tax_Id=(?P<species>[^\ ]+).*?"` -> A format like ">Q32MB2 TREMBL:Q32MB2 Tax_Id=9606 Gene_Symbol=KRT73"
    - `fasta_title_regex=[regex1, regex2]` -> Try to parse title sequentially until both "protein" and "species" are found. If "protein" can not be found by any regex, will raise error. If "species" can not be found, it will be "unknown", and "protein" will be the first matched one.

In [6]:
parsed_fasta = lipana.parse_fasta(
    fasta_path=workspace.joinpath("human_yeast_contam.fasta"),
    contam_fasta_path=workspace.joinpath("contam.fasta"),
    contaminations=None,
    fasta_title_regex=None,  # Set one or more regex if the title is not UniProt format or there are mixed formats in the FASTA files
    gen_species_to_concat_seqs=True,  # This parameter is useful to annotate species for peptides via a whole protein sequence map, but is time-consuming
    workspace=workspace,
    resume=True,
    write_parsed_fasta=True,
)

### Load and annotate a search report

- Now, load and annotate the search report using the previously set up `ExperimentLayout` and `ParsedFasta`

In [7]:
# The load_search_report methods for DIA-NN and Spectronaut share parameters:
# lipana.DIANNReport.load_search_report(...)
# lipana.SpectronautReport.load_search_report(...)

# Example loading a DIA-NN search report, showing a subset of available parameters
report = lipana.DIANNReport.load_search_report(
    workspace.joinpath("example_diann_report.txt"),
    exp_layout=exp_layout,
    parsed_fasta=parsed_fasta,
    do_species_annotation=True,  # Set this to True if the experiment includes multiple species
    pre_annotation_filter=(
        (pl.col("Q.Value") < 0.01)
        & (pl.col("Lib.PG.Q.Value") < 0.01)
        & (pl.col("Protein.Group").is_not_null())
        & (pl.col("Precursor.Quantity").is_not_null())
        & (pl.col("Precursor.Quantity") > 1.1)
    ),  # The filters defined here will be applied to the report after loading immediately, and here the 5 rules are default for DIA-NN (if this parameter is not given, these rules will also be applied); for Spectronaut, they are (
    #     (pl.col("PG.ProteinGroups").is_not_null())
    #     & (pl.col("FG.Quantity").is_not_null())
    #     & (pl.col("FG.Quantity") > 1.1)
    # )
    post_annotation_filter=None,  # This parameter can be used to filter the report after annotation
    restricted_cut_sites=("K", "R"),
    expand_to_cut_site_level=True,  # This will make each cut site one row, which means each precursor row will be duplicated
    # modification_map=None,  # This parameter can be either None or a map from software used modification to UniMod. By default, DIA-NN report has None because it use UniMod as default; for `SpectronautReport`, this parameter has a default value as {
    #     "[Acetyl (Protein N-term)]": "(UniMod:1)",
    #     "[Carbamidomethyl (C)]": "(UniMod:4)",
    #     "[Oxidation (M)]": "(UniMod:35)",
    # }
)
# The variable `report` is now a `SearchReport` object, and basic annotations have been done
report

Search report object with main report shape: (113057, 81)
Number of quantification data: 0
Number of statistics results: 0
Workspace: /home/data/LiPAna/example_data

## Preparation of Quantification Matrices

- A newly created `SearchReport` object initially contains only a preprocessed search report and associated metadata, and in this section, some quantification matrices will be added to it
- The quantification process can simply involve pivoting the report or aggregating data using methods like topN or maxLFQ

### Use raw reported quantity values

- This approach converts the report into a quantification matrix by using the raw quantity values reported by the software as matrix cells. It is particularly useful for generating precursor-level or protein group-level matrices, since most software will report the quantity values on these two levels
- Additionally, this step demonstrates how to manually attach a customized quantification report to the main report using the `attach_quant_data` method

In [8]:
quant_data = report.attach_quant_data(
    lipana.convert_long_report_to_wide(
        report,
        index_col="precursor",
        column_col="run",
        value_col="precursor_quantity",
        do_log_scale=2,
        pl_filter=(
            ~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)
        ),  # This parameter filters out those peptides mapped to multiple species
        do_unique=True,
    ),
    # These two parameters will be the name of this quantification report, and can be used to retrieve it from the main report object
    # Though they can be named as any strings, it is recommended to use the same name as the entry level and the actually used quantification method
    entry_level="precursor",
    quant_method="precursor_quantity",
)

# To retrieve this object from the main report object, use the following command
quant_data = report.get_quant_data("precursor", "precursor_quantity")


### maxLFQ and topN Methods

- Both maxLFQ and topN methods utilize the same interface through the `SearchReport.construct_and_attach_quant_data` method, featuring convenient designs to handle more complex scenarios
- The following examples will concentrate on maxLFQ; topN can be configured by setting parameters like `method="top3"` or `method="top1"`, etc.


#### Peptide matrix from precursor MS2 via maxLFQ

- This example will do maxLFQ on precursor quantities to get a peptide level matrix

In [9]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="peptide_quant_by_maxlfq_from_prec",  # This parameter will be the name of quantification method for this quantification report, and can be used to retrieve it from the main report object
    method="maxlfq",
    filter_condition=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
    run_col="run",
    primary_entry_col="stripped_peptide",  # This parameter not only define the primary (target) entry level, but also the first name to retrieve the generated matrix from the main report object
    low_level_entry_col="precursor",
    base_quant_col="precursor_quantity",
    require_expansion=False,
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmpvk98unh_.txt
Removing low intensities...

Output to /tmp/tmpvk98unh_.txt-iq_output.txt



Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmpvk98unh_.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 55844
# quantitative values after filtering = 55844

# samples  = 9
# proteins = 9577
nrow = 55844, # proteins = 9577, # samples = 9
Using 95 threads...
0%
5%
10%
15%
20%
25%
30%
35%
40%
45%
50%
56%
61%
66%
71%
76%
82%
87%
92%
99%
Completed.


#### Peptide matrix from prec MS1 and prec MS2 via maxLFQ

- The quantities of base level entries can also be from multiple sources
- The following example will use maxLFQ to aggregate quantities from two levels: precursor MS1 and precursor MS2

In [10]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="peptide_quant_by_maxlfq_from_ms1ms2",
    method="maxlfq",
    filter_condition=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
    run_col="run",
    primary_entry_col="stripped_peptide",
    low_level_entry_col=("precursor", "precursor"),
    base_quant_col=("precursor_quantity", "precursor_quantity_ms1"),
    require_expansion=(False, False),
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmp44pu8q7v.txt
Removing low intensities...




Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmp44pu8q7v.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 111685
# quantitative values after filtering = 111685

# samples  = 9
# proteins = 9577
nrow = 111685, # proteins = 9577, # samples = 9
Using 95 threads...
0%
5%
10%
15%
20%
25%
32%
37%
42%
48%
54%
59%
65%
70%
75%
81%
86%
91%
98%
Completed.


Output to /tmp/tmp44pu8q7v.txt-iq_output.txt


#### Peptide matrix from prec MS1, prec MS2, and fragments, via maxLFQ

- The following example shows a more complicated case, which uses maxLFQ to aggregate quantities from three levels: precursor MS1, precursor MS2, and fragments (the fragment quantity values in the DIA-NN tsv report is in one table cell joined by ";" for each row)


In [11]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="peptide_quant_by_maxlfq_from_ms1ms2frag",
    method="maxlfq",
    filter_condition=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
    run_col="run",
    primary_entry_col="stripped_peptide",
    low_level_entry_col=("precursor", "precursor", "Fragment.Info"),
    base_quant_col=("precursor_quantity", "precursor_quantity_ms1", "Fragment.Quant.Raw"),
    require_expansion=(False, False, ";"),
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmpw66_1tn6.txt



Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmpw66_1tn6.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 701484
# quantitative values after filtering = 701484

# samples  = 9
# proteins = 9577


Removing low intensities...



nrow = 701484, # proteins = 9577, # samples = 9
Using 95 threads...
0%
5%
10%
16%
22%
27%
32%
37%
43%
49%
54%
59%
64%
70%
76%
81%
86%
92%
97%
Completed.


Output to /tmp/tmpw66_1tn6.txt-iq_output.txt


#### Non-restricted cut site from precursor MS2+MS1

- Sometimes, the quantity values should be aggregated from a filtered report
- The following in-one-step example will generate a cut site level matrix, where the cut sites are only non-restricted
  - Whether a cut site is restricted has been annotated via the parameter `restricted_cut_sites` in method `load_search_report` (in this doc, it's `lipana.DIANNReport.load_search_report(..., restricted_cut_sites=("K", "R"), ...)`)
  - Besides the filtering for site annotation `(~pl.col("cut_site_is_restricted"))`, there is also a filter  for peptide, because the C-term of a cut site can be the second residue of a protein, which would be surrounded by first "M" and the third any residue. Here the cut site will be annotated as "non-restricted" if the third residue is K/R, but it's hard to say if the cleavage between protein N-term "M" and the second residue is caused during translation or by a non-restricted enzyme. Further restraining peptide to be semi-specific will address this problem by filtering out such peptides at the protein N-term.

In [12]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="nonrestricted_cut_site_by_maxlfq_from_prec",
    method="maxlfq",
    filter_condition=(
        (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
        & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
        & (~pl.col("cut_site_is_restricted"))
    ),
    run_col="run",
    primary_entry_col="cut_site",
    low_level_entry_col=("precursor", "precursor"),
    base_quant_col=("precursor_quantity", "precursor_quantity_ms1"),
    require_expansion=(False, False),
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)


Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmp0dawcd1g.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 52579
# quantitative values after filtering = 52579

# samples  = 9
# proteins = 4116
nrow = 52579, # proteins = 4116, # samples = 9
Using 95 threads...
1%
6%
12%
17%
22%
27%
32%
37%
42%
47%
52%
58%
64%
72%
77%
Completed.


MaxLFQ estimation via iq for /tmp/tmp0dawcd1g.txt
Removing low intensities...

Output to /tmp/tmp0dawcd1g.txt-iq_output.txt


### Utilities on quantification report

- The quantification matrix generated from the above sections is actually an object `EntryQuantificationReport`, which has many handful functions

In [13]:
# If the generated object is not explicitly assigned to a variable, can use the following way to get from main report object
quant_data = report.get_quant_data("precursor", "precursor_quantity")

# The following three methods will attach new columns to the dataframe stored in the quantification report object
quant_data.calc_cv(cond="all", min_reps=3, temp_reverse_log_scale=2, new_colname_pattern="{cond}_cv_{min_reps}reps")
quant_data.count_detected_replicates(new_colname_pattern="{cond}_detected_reps")
quant_data.calc_ratio(
    base_cond="H5_Y100",
    is_log=True,
    temp_reverse_log_scale=2,
    div_method="agg_and_divide",
    agg_method="mean",
    new_colname_pattern="ratio_{cond1}_to_{cond2}",
)

precursor,20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163,20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164,20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206,20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155,20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154,20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158,20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220,20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210,20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157,H2d5_Y100_cv_3reps,H5_Y100_cv_3reps,H25_Y100_cv_3reps,H2d5_Y100_detected_reps,H5_Y100_detected_reps,H25_Y100_detected_reps,ratio_H2d5_Y100_to_H5_Y100,ratio_H25_Y100_to_H5_Y100
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32
"""FSTEHRTELLTEALR/3""",14.709242,13.7669,null,null,null,null,14.288789,null,null,NaN,NaN,31.741871,0,0,3,NaN,NaN
"""(UniMod:1)AGLTVRDPAVDR/2""",13.883706,14.774942,null,null,null,null,14.36131,null,null,NaN,NaN,30.22172,0,0,3,NaN,NaN
"""LIESSNLEMEIIPNQK/2""",16.247179,16.857989,13.776511,14.950796,13.97397,14.926452,15.559869,14.37336,15.433312,46.817608,35.862591,43.365429,3,3,3,-0.644731,1.341784
"""KTASEVPAHL/2""",14.119793,14.213616,14.163261,13.956965,13.849289,14.365936,13.940348,13.951865,13.155161,11.220558,39.130417,9.476161,3,3,3,0.088797,0.18868
"""NGFILDGFPR/2""",18.556313,18.90354,19.189005,18.773484,18.508408,18.784788,19.578332,19.374305,18.605135,24.196247,29.262712,37.243092,3,3,3,-0.108961,0.116536
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""LNPWNDNKK/2""",null,null,null,null,null,null,13.488453,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN
"""LMADVVEEETR/2""",null,null,null,null,null,null,18.325743,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN
"""VEELERW/2""",null,null,null,null,null,null,14.556121,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN


- Because main report and quantification report are related, the annotation and filtering of quantification report can be done by borrowing data from the main report
- The following method will return a new dataframe instead of updating the dataframe stored in the object

In [ ]:
report.get_quant_data(
    entry_name="precursor",
    quant_name="precursor_quantity",
    main_df_entry_filter=(
        (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
        & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
    ),
    quant_df_filter=None,
    annotation_cols="protein_group",
)

precursor,20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163,20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164,20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206,20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155,20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154,20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158,20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220,20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210,20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157,H2d5_Y100_cv_3reps,H5_Y100_cv_3reps,H25_Y100_cv_3reps,H2d5_Y100_detected_reps,H5_Y100_detected_reps,H25_Y100_detected_reps,ratio_H2d5_Y100_to_H5_Y100,ratio_H25_Y100_to_H5_Y100,protein_group
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32,str
"""KTASEVPAHL/2""",14.119793,14.213616,14.163261,13.956965,13.849289,14.365936,13.940348,13.951865,13.155161,11.220558,39.130417,9.476161,3,3,3,0.088797,0.18868,"""P38219"""
"""AAIVEIIDQKK/2""",13.790874,14.064759,14.048119,13.116001,13.630545,13.167408,13.846001,14.357204,13.977925,31.426075,38.366875,10.268633,3,3,3,-0.266841,-0.008983,"""P36105"""
"""DKEWMPVTK/2""",13.262836,13.824716,null,null,null,null,14.200207,null,null,NaN,NaN,31.265383,0,0,3,NaN,NaN,"""P15880"""
"""TIPQAEKLDQM/2""",18.202073,18.419359,18.406268,18.381875,18.423939,18.390974,18.429694,18.630445,18.463898,1.462301,8.628253,8.671094,3,3,3,-0.09449,-0.144495,"""P07170"""
"""IDDSDIPNNEMK/2""",12.381129,12.647393,12.342149,12.921892,12.679637,12.863412,13.162064,12.42859,12.547918,19.754705,15.997437,28.332909,3,3,3,0.042018,0.142178,"""P51862"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""RPHDYQPLPGMSENPSVYVPGVVST/3""",null,9.93736,null,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN,"""P26368"""
"""LNPWNDNKK/2""",null,null,null,null,null,null,13.488453,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN,"""Q9H6S0"""
"""VEELERW/2""",null,null,null,null,null,null,14.556121,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN,"""O75935"""


In [ ]:
quant_data = report.get_quant_data("precursor", "precursor_quantity")
quant_data.attach_annotation_via_entry(
    "protein_group", persist=False
)  # Set `persistent` to True will update the dataframe stored in the object
quant_data.filter_entry_by_main_report(
    main_report_filter=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
    & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
)


precursor,20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163,20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164,20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206,20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155,20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154,20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158,20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220,20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210,20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157,H2d5_Y100_cv_3reps,H5_Y100_cv_3reps,H25_Y100_cv_3reps,H2d5_Y100_detected_reps,H5_Y100_detected_reps,H25_Y100_detected_reps,ratio_H2d5_Y100_to_H5_Y100,ratio_H25_Y100_to_H5_Y100
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32
"""KTASEVPAHL/2""",14.119793,14.213616,14.163261,13.956965,13.849289,14.365936,13.940348,13.951865,13.155161,11.220558,39.130417,9.476161,3,3,3,0.088797,0.18868
"""AAIVEIIDQKK/2""",13.790874,14.064759,14.048119,13.116001,13.630545,13.167408,13.846001,14.357204,13.977925,31.426075,38.366875,10.268633,3,3,3,-0.266841,-0.008983
"""DKEWMPVTK/2""",13.262836,13.824716,null,null,null,null,14.200207,null,null,NaN,NaN,31.265383,0,0,3,NaN,NaN
"""TIPQAEKLDQM/2""",18.202073,18.419359,18.406268,18.381875,18.423939,18.390974,18.429694,18.630445,18.463898,1.462301,8.628253,8.671094,3,3,3,-0.09449,-0.144495
"""IDDSDIPNNEMK/2""",12.381129,12.647393,12.342149,12.921892,12.679637,12.863412,13.162064,12.42859,12.547918,19.754705,15.997437,28.332909,3,3,3,0.042018,0.142178
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""RPHDYQPLPGMSENPSVYVPGVVST/3""",null,9.93736,null,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN
"""LNPWNDNKK/2""",null,null,null,null,null,null,13.488453,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN
"""VEELERW/2""",null,null,null,null,null,null,14.556121,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN


## Statistics

- The `lipana` package offers several built-in functions that users can chain together to create custom statistical pipelines. For convenience, the package also includes a wrapper function offering three predefined pipelines.
- This section demonstrates how to utilize these predefined pipelines for performing statistical tests.
- Note (missing value): The `lipana.do_stats_pipe_direct` function includes an optional parameter, `mv_config`, which defaults to `None`.
  - Default Behavior (`mv_config=None`): If `mv_config` is not specified or set to `None`, the input data is filtered directly based on the `min_rep_count` threshold before any statistical testing.
  - Using Missing Value Handlers: You can set `mv_config` to an instance (or a list of instances) of `AbstractPairwiseMissingValueHandler`. Examples include: `FullEmptyFillingMissingValueHandler(min_rep_count=3)`,  `NormalDistSamplingMissingValueHandler(min_rep_count=3, do_imputation_for_rep_range=(1, 2), log_scale_minmax_diff=1)`. When a handler is provided, missing value processing occurs first, and the `min_rep_count` requirement is checked afterward.
  - Example Handler Usage: The examples below will use `FullEmptyFillingMissingValueHandler(min_rep_count=2)`. This handler imputes a value of 0 (on the log-scale) for entries where one condition has no detected replicates, provided the other condition in the pair has at least 2 detected replicates (`>= min_rep_count`).
  - Disabling Imputation: To disable any specific missing value filling/imputation while still potentially using the handler structure, set `mv_config` to `lipana.NullMissingValueHandler()`. Filtering will still rely on `min_rep_count` after the (null) handling step.
- Note (aggregation): The `lipana.do_stats_pipe_agg` function uses the `pipeline` parameter to control how results are aggregated from a base level (e.g., peptides) to a target level (e.g., proteins or cut sites). Specify one of the following options for the `pipeline` parameter:
  - `"sel_min_p"`: Selects the single base-level entry with the minimum p-value for each target-level entry.
  - `"sel_min2_p"`, `"sel_min3_p"`, ...: Selects the top 2, 3, etc., base-level entries with the lowest p-values for each target-level entry.
  - `"combine_p"`: Combines p-values from all associated base-level entries for each target-level entry using Fisher's method. The reported fold change (FC) will be the median FC of the combined base-level entries.
  - Pipelines with `_direction_check` Suffix (e.g., `"sel_min_p_direction_check"`, `"combine_p_direction_check"`): These pipelines first assess the directionality (sign of the t-statistic in the predefined pipelines) of fold changes for all base entries mapping to a target entry. They determine if there's a majority direction (positive or negative). Only the base entries matching this majority direction are considered for the subsequent selection or combination step.

### Example 1: Test on Fully-Specific Stripped Peptides

This example performs hypothesis tests directly on the quantification data for fully-specific stripped peptides. The results remain at the peptide level.

In [16]:
design = lipana.ComparisonDesign(exp_layout, (("H2d5_Y100", "H5_Y100"), ("H25_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = (pl.col("peptide_enzymatic_specificity") == "fully_specific") & (
    pl.col("mapped_species_from_peptide").is_in(("HUMAN", "YEAST"))
)

stats_fully_pep = lipana.stats.pipe.do_stats_pipe_direct(
    qdf=report.get_quant_data(
        entry_name="stripped_peptide",
        quant_name="peptide_quant_by_maxlfq_from_ms1ms2",
        main_df_entry_filter=filters,
    ),  # Note here no need to call `.df` because the requirement of filtering will make this function return a dataframe, instead of a `EntryQuantificationReport` object
    design=design,
    entry_level="stripped_peptide",
    mv_config=lipana.FullEmptyFillingMissingValueHandler(2),
    min_rep_count=2,
    annotation_df=report.df.filter(filters),
    group_entry_level="protein_group",  # Note this parameter is given because FDR control also can be done on the protein level
    annotation_cols=[
        "peptide_start_position",
        "peptide_end_position",
        "prev_aa",
        "next_aa",
        "peptide_n_term_aa",
        "peptide_c_term_aa",
        "nterm_enzymatic_specificity",
        "cterm_enzymatic_specificity",
        "peptide_enzymatic_specificity",
        "n_cut_site",
        "c_cut_site",
    ],
    return_chains=False,
)
sleep(0.1)
print(f"First 5 rows of the result (total size: {stats_fully_pep.shape}):")
stats_fully_pep.head(5)

design.pairwise_comparisons = (('H2d5_Y100', 'H5_Y100'), ('H25_Y100', 'H5_Y100'))


Quantification file: /tmp/tmpa55k2kkp.txt
All defined comparison pairs: c("H2d5_Y100", "H5_Y100")
Available comparison pairs: c("H2d5_Y100", "H5_Y100")
Comparing H2d5_Y100 vs H5_Y100
Output to /tmp/tmpa55k2kkp.txt-limma_output.txt
Quantification file: /tmp/tmpd9vn2h4r.txt
All defined comparison pairs: c("H25_Y100", "H5_Y100")
Available comparison pairs: c("H25_Y100", "H5_Y100")
Comparing H25_Y100 vs H5_Y100
Output to /tmp/tmpd9vn2h4r.txt-limma_output.txt


First 5 rows of the result (total size: (10126, 27)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str
"""RIEYIEAR""",-14.269712,7.134856,-141.989875,1.2210e-12,5.5491e-10,15.865813,"""H2d5_Y100_vs_H5_Y100""",-14.270495,0,3,"""full_empty""",true,5.5491e-10,"""Q8WUW1""",1.2210e-12,60,67,"""R""","""V""","""R""","""R""","""restricted""","""restricted""","""fully_specific""","""Q8WUW1-59_60""","""Q8WUW1-67_68"""
"""AGAFEHLPSLR""",-13.485006,6.742503,-136.835559,1.5543e-12,5.5491e-10,15.815418,"""H2d5_Y100_vs_H5_Y100""",-13.485344,0,3,"""full_empty""",true,5.5491e-10,"""Q13641""",3.1085e-12,135,145,"""R""","""Q""","""A""","""R""","""restricted""","""restricted""","""fully_specific""","""Q13641-134_135""","""Q13641-145_146"""
"""QMADTGKLNTLLQR""",-13.452959,6.72648,-136.678498,1.5659e-12,5.5491e-10,15.813804,"""H2d5_Y100_vs_H5_Y100""",-13.453273,0,3,"""full_empty""",true,5.5491e-10,"""Q00839""",2.8187e-11,545,558,"""K""","""A""","""Q""","""R""","""restricted""","""restricted""","""fully_specific""","""Q00839-544_545""","""Q00839-558_559"""
"""LQDLQGEKDALHSEK""",-13.04591,6.522955,-132.239542,1.9425e-12,5.5491e-10,15.766083,"""H2d5_Y100_vs_H5_Y100""",-13.046274,0,3,"""full_empty""",true,5.5491e-10,"""O60610""",2.1367e-11,513,527,"""K""","""Q""","""L""","""K""","""restricted""","""restricted""","""fully_specific""","""O60610-512_513""","""O60610-527_528"""
"""LTEELKEQMK""",-12.956682,6.478341,-127.631859,2.4483e-12,5.5491e-10,15.711929,"""H2d5_Y100_vs_H5_Y100""",-12.957698,0,3,"""full_empty""",true,5.5491e-10,"""Q14683""",5.0468e-12,681,690,"""R""","""A""","""L""","""K""","""restricted""","""restricted""","""fully_specific""","""Q14683-680_681""","""Q14683-690_691"""


### Example 2: Test on Semi-Specific Stripped Peptides and Aggregate to Cut Site Level

This example demonstrates a multi-step process:
1.  Selects semi-specific peptides from search reports.
2.  Performs hypothesis tests on these selected peptides.
3.  Annotates the results with non-restricted cut sites.
4.  Aggregates the peptide-level p-values and fold changes to the cut site level.


In [17]:
design = lipana.ComparisonDesign(exp_layout, (("H2d5_Y100", "H5_Y100"), ("H25_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = (pl.col("peptide_enzymatic_specificity") == "semi_specific") & (
    pl.col("mapped_species_from_peptide").is_in(("HUMAN", "YEAST"))
)

stats_semi_pep = lipana.stats.pipe.do_stats_pipe_direct(
    qdf=report.get_quant_data(
        entry_name="stripped_peptide",
        quant_name="peptide_quant_by_maxlfq_from_ms1ms2",
        main_df_entry_filter=filters,
    ),  # Note here no need to call `.df` because the requirement of filtering will make this function return a dataframe, instead of a `EntryQuantificationReport` object
    design=design,
    entry_level="stripped_peptide",
    mv_config=lipana.FullEmptyFillingMissingValueHandler(2),
    min_rep_count=2,
    annotation_df=report.df.filter(filters),
    group_entry_level="protein_group",  # Note this parameter is given because FDR control also can be done on the protein level
    annotation_cols=[
        "peptide_start_position",
        "peptide_end_position",
        "prev_aa",
        "next_aa",
        "peptide_n_term_aa",
        "peptide_c_term_aa",
        "nterm_enzymatic_specificity",
        "cterm_enzymatic_specificity",
        "peptide_enzymatic_specificity",
        "n_cut_site",
        "c_cut_site",
    ],
    return_chains=False,
)
sleep(0.1)
print(f"First 5 rows of the result (total size: {stats_semi_pep.shape}):")
stats_semi_pep.head(5)

design.pairwise_comparisons = (('H2d5_Y100', 'H5_Y100'), ('H25_Y100', 'H5_Y100'))


Quantification file: /tmp/tmpuerov96f.txt
All defined comparison pairs: c("H2d5_Y100", "H5_Y100")
Available comparison pairs: c("H2d5_Y100", "H5_Y100")
Comparing H2d5_Y100 vs H5_Y100
Output to /tmp/tmpuerov96f.txt-limma_output.txt
Quantification file: /tmp/tmpssztdyla.txt
All defined comparison pairs: c("H25_Y100", "H5_Y100")
Available comparison pairs: c("H25_Y100", "H5_Y100")
Comparing H25_Y100 vs H5_Y100
Output to /tmp/tmpssztdyla.txt-limma_output.txt


First 5 rows of the result (total size: (8930, 27)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str
"""KQVEQLEK""",-15.259082,7.629541,-131.992226,3.3089e-13,2.1076e-10,16.82335,"""H2d5_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,2.1076e-10,"""Q14152""",6.6178e-12,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680"""
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-131.814803,3.3404e-13,2.1076e-10,16.821376,"""H2d5_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,2.1076e-10,"""P08865""",2.3007e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116"""
"""ADHQPLTEA""",-14.459833,7.229916,-129.255177,3.8345e-13,2.1076e-10,16.792104,"""H2d5_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,2.1076e-10,"""P08865""",2.3007e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138"""
"""ATSISPEQCIK""",-14.37804,7.18902,-127.536615,4.2132e-13,2.1076e-10,16.771583,"""H2d5_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,2.1076e-10,"""O75122""",1.6853e-12,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182"""
"""QQNQDPEVFVQPK""",-14.076481,7.03824,-123.013286,5.4318e-13,2.1076e-10,16.713963,"""H2d5_Y100_vs_H5_Y100""",-14.077208,0,3,"""full_empty""",true,2.1076e-10,"""Q9NPI6""",4.3455e-12,464,476,"""M""","""V""","""Q""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q9NPI6-463_464""","""Q9NPI6-476_477"""


In [18]:
stats_cut_site = lipana.stats.pipe.do_stats_pipe_agg(
    qdf=stats_semi_pep,
    base_entry="stripped_peptide",
    target_entry="cut_site",
    group_entry="protein_group",
    annotation_df=report.df.filter(~pl.col("cut_site_is_restricted")),
    annotation_cols=[
        "cut_site",
        "cut_site_n_aa",
        "cut_site_c_aa",
        "cut_site_on_term",
        "cut_site_is_restricted",
    ],
    pipeline="combine_p_direction_check",
)
print(f"First 5 rows of the result (total size: {stats_cut_site.shape}):")
stats_cut_site.head(5)

/home/data/LiPAna/lipana/stats/infer.py:102: RuntimeWarning: All-NaN slice encountered
  func_out = func(x)


First 5 rows of the result (total size: (8998, 43)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,cut_site,cut_site_n_aa,cut_site_c_aa,cut_site_on_term,cut_site_is_restricted,row_sign,n_detection_in_group,is_group_sign_balanced,group_major_sign,sign_filter_passed,log2_fc_combined,log2_fc_limma_combined,pvalue_combined,first_nonnan_in_combined,adj_pvalue_combined_exp_wise,adj_pvalue_combined_group_wise
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,f64,u32,bool,str,bool,f32,f64,f64,bool,f64,f64
"""KQVEQLEK""",-15.259082,7.629541,-131.992226,3.3089e-13,2.1076e-10,16.82335,"""H2d5_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,2.1076e-10,"""Q14152""",6.6178e-12,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680""","""Q14152-671_672""","""A""","""K""","""n""",false,-1.0,1,false,"""neg""",true,-15.260145,-15.259082,3.3089e-13,true,5.1673e-11,6.6178e-12
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-131.814803,3.3404e-13,2.1076e-10,16.821376,"""H2d5_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,2.1076e-10,"""P08865""",2.3007e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116""","""P08865-115_116""","""A""","""F""","""c""",false,-1.0,1,false,"""neg""",true,-15.044158,-15.043514,3.3404e-13,false,NaN,NaN
"""ADHQPLTEA""",-14.459833,7.229916,-129.255177,3.8345e-13,2.1076e-10,16.792104,"""H2d5_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,2.1076e-10,"""P08865""",2.3007e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138""","""P08865-137_138""","""A""","""S""","""c""",false,-1.0,2,false,"""neg""",true,-14.459851,-14.459833,3.8345e-13,false,NaN,NaN
"""ATSISPEQCIK""",-14.37804,7.18902,-127.536615,4.2132e-13,2.1076e-10,16.771583,"""H2d5_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,2.1076e-10,"""O75122""",1.6853e-12,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182""","""O75122-1170_1171""","""L""","""A""","""n""",false,-1.0,1,false,"""neg""",true,-14.378295,-14.37804,4.2132e-13,false,NaN,NaN
"""QQNQDPEVFVQPK""",-14.076481,7.03824,-123.013286,5.4318e-13,2.1076e-10,16.713963,"""H2d5_Y100_vs_H5_Y100""",-14.077208,0,3,"""full_empty""",true,2.1076e-10,"""Q9NPI6""",4.3455e-12,464,476,"""M""","""V""","""Q""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q9NPI6-463_464""","""Q9NPI6-476_477""","""Q9NPI6-463_464""","""M""","""Q""","""n""",false,-1.0,1,false,"""neg""",true,-14.077208,-14.076481,5.4318e-13,false,NaN,NaN


### Test on All Stripped Peptides and Aggregate to Protein Group Level

This example performs hypothesis tests on all stripped peptides (regardless of specificity) and then aggregates the results to the protein group level. The aggregation method used here selects the base peptide(s) with the minimum p-value(s) for each protein group (e.g., using `"sel_min_p"` or `"sel_min2_p"`).

In [19]:
design = lipana.ComparisonDesign(exp_layout, (("H2d5_Y100", "H5_Y100"), ("H25_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = pl.col("mapped_species_from_peptide").is_in(("HUMAN", "YEAST"))

stats_all_pep = lipana.stats.pipe.do_stats_pipe_direct(
    qdf=report.get_quant_data(
        entry_name="stripped_peptide",
        quant_name="peptide_quant_by_maxlfq_from_ms1ms2",
        main_df_entry_filter=filters,
    ),  # Note here no need to call `.df` because the requirement of filtering will make this function return a dataframe, instead of a `EntryQuantificationReport` object
    design=design,
    entry_level="stripped_peptide",
    mv_config=lipana.FullEmptyFillingMissingValueHandler(2),
    min_rep_count=2,
    annotation_df=report.df.filter(filters),
    group_entry_level="protein_group",  # Note this parameter is given because FDR control also can be done on the protein level
    annotation_cols=[
        "peptide_start_position",
        "peptide_end_position",
        "prev_aa",
        "next_aa",
        "peptide_n_term_aa",
        "peptide_c_term_aa",
        "nterm_enzymatic_specificity",
        "cterm_enzymatic_specificity",
        "peptide_enzymatic_specificity",
        "n_cut_site",
        "c_cut_site",
    ],
    return_chains=False,
)
sleep(0.1)
print(f"First 5 rows of the result (total size: {stats_all_pep.shape}):")
stats_all_pep.head(5)

design.pairwise_comparisons = (('H2d5_Y100', 'H5_Y100'), ('H25_Y100', 'H5_Y100'))


Quantification file: /tmp/tmp0sywt2cb.txt
All defined comparison pairs: c("H2d5_Y100", "H5_Y100")
Available comparison pairs: c("H2d5_Y100", "H5_Y100")
Comparing H2d5_Y100 vs H5_Y100
Output to /tmp/tmp0sywt2cb.txt-limma_output.txt
Quantification file: /tmp/tmpmtq5r82y.txt
All defined comparison pairs: c("H25_Y100", "H5_Y100")
Available comparison pairs: c("H25_Y100", "H5_Y100")
Comparing H25_Y100 vs H5_Y100
Output to /tmp/tmpmtq5r82y.txt-limma_output.txt


First 5 rows of the result (total size: (19056, 27)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-141.146103,5.5789e-13,4.1049e-10,16.326055,"""H2d5_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,4.1049e-10,"""P08865""",5.5806e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116"""
"""KQVEQLEK""",-15.259082,7.629541,-140.988505,5.6212e-13,4.1049e-10,16.324581,"""H2d5_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,4.1049e-10,"""Q14152""",3.4851e-11,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680"""
"""ADHQPLTEA""",-14.459833,7.229916,-138.955631,6.2007e-13,4.1049e-10,16.305165,"""H2d5_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,4.1049e-10,"""P08865""",5.5806e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138"""
"""ATSISPEQCIK""",-14.37804,7.18902,-136.894954,6.8593e-13,4.1049e-10,16.2847,"""H2d5_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,4.1049e-10,"""O75122""",5.4875e-12,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182"""
"""RIEYIEAR""",-14.269712,7.134856,-133.198888,8.2526e-13,4.1049e-10,16.245885,"""H2d5_Y100_vs_H5_Y100""",-14.270495,0,3,"""full_empty""",true,4.1049e-10,"""Q8WUW1""",8.2526e-13,60,67,"""R""","""V""","""R""","""R""","""restricted""","""restricted""","""fully_specific""","""Q8WUW1-59_60""","""Q8WUW1-67_68"""


In [ ]:
stats_protein_group = lipana.stats.pipe.do_stats_pipe_agg(
    qdf=stats_all_pep,
    base_entry="stripped_peptide",
    target_entry="protein_group",
    group_entry=None,
    annotation_df=None,
    annotation_cols=None,
    pipeline="sel_min_p",
)
print(f"First 5 rows of the result (total size: {stats_protein_group.shape}):")
stats_protein_group.head(5)

First 5 rows of the result (total size: (19056, 29)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,top_k_selected,adj_pvalue_combined_exp_wise
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,bool,f64
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-141.146103,5.5789e-13,4.1049e-10,16.326055,"""H2d5_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,4.1049e-10,"""P08865""",5.5806e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116""",true,4.1049e-10
"""KQVEQLEK""",-15.259082,7.629541,-140.988505,5.6212e-13,4.1049e-10,16.324581,"""H2d5_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,4.1049e-10,"""Q14152""",3.4851e-11,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680""",true,4.1049e-10
"""ADHQPLTEA""",-14.459833,7.229916,-138.955631,6.2007e-13,4.1049e-10,16.305165,"""H2d5_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,4.1049e-10,"""P08865""",5.5806e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138""",false,NaN
"""ATSISPEQCIK""",-14.37804,7.18902,-136.894954,6.8593e-13,4.1049e-10,16.2847,"""H2d5_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,4.1049e-10,"""O75122""",5.4875e-12,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182""",true,4.1049e-10
"""RIEYIEAR""",-14.269712,7.134856,-133.198888,8.2526e-13,4.1049e-10,16.245885,"""H2d5_Y100_vs_H5_Y100""",-14.270495,0,3,"""full_empty""",true,4.1049e-10,"""Q8WUW1""",8.2526e-13,60,67,"""R""","""V""","""R""","""R""","""restricted""","""restricted""","""fully_specific""","""Q8WUW1-59_60""","""Q8WUW1-67_68""",true,4.1049e-10


## Utilities

### Retrieve annotation from main report for quantification data or statistic result

- To reduce the additional use of memory, all annotations are generally only stored in the report
- To retrieve these information from the main report, the table join can be done on the dataframe that needs some information and the main report `report.df`
- To handle some potential complicated cases, there is also a function `lipana.attach_annotation_from_other_df` to do this action
- Note: It is highly recommended to attach the desired annotations to the quantification dataframe and thoroughly verify their accuracy before proceeding with statistical analysis. This verification step is crucial because annotations can sometimes exhibit one-to-many (1:n) mappings. For example, reports from analysis software might list the same peptide sequence on multiple rows, associating it with different protein groups or genes in each instance.

In [21]:
# Retrieve some protein annotations for the stats result
stats_protein_group = lipana.attach_annotation_from_other_df(
    df=stats_protein_group,
    other_df=report.df,
    annotation_cols=[
        "Protein.Names",
        "Genes",
        "First.Protein.Description",
    ],
    on="protein_group",
    pre_filter=None,
    unique_on_key_only=False,
    method="leftjoin",
)
stats_protein_group.sort("protein_group").unique("protein_group").head(5)

stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,top_k_selected,adj_pvalue_combined_exp_wise,Protein.Names,Genes,First.Protein.Description
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,bool,f64,str,str,str
"""RPPGEPLQGF""",null,null,null,null,null,null,"""H2d5_Y100_vs_H5_Y100""",NaN,0,1,"""""",false,NaN,"""A1L020""",NaN,443,452,"""R""","""S""","""R""","""F""","""restricted""","""non_restricted""","""semi_specific""","""A1L020-442_443""","""A1L020-452_453""",false,NaN,"""MEX3A_HUMAN""","""MEX3A""","""RNA-binding protein MEX3A"""
"""FANMPVESELER""",null,null,null,null,null,null,"""H2d5_Y100_vs_H5_Y100""",NaN,0,0,"""""",false,NaN,"""A4D1U4""",NaN,136,147,"""C""","""G""","""F""","""R""","""non_restricted""","""restricted""","""semi_specific""","""A4D1U4-135_136""","""A4D1U4-147_148""",false,NaN,"""DEN11_HUMAN""","""DENND11""","""DENN domain-containing protein…"
"""NLAEDGETVAR""",null,null,null,null,null,null,"""H2d5_Y100_vs_H5_Y100""",NaN,0,1,"""""",false,NaN,"""A6NED2""",NaN,248,258,"""R""","""E""","""N""","""R""","""restricted""","""restricted""","""fully_specific""","""A6NED2-247_248""","""A6NED2-258_259""",false,NaN,"""RCCD1_HUMAN""","""RCCD1""","""RCC1 domain-containing protein…"
"""AALQEEEQASK""",-13.009611,6.504806,-81.286978,1.1566e-8,1.7648e-7,8.948449,"""H2d5_Y100_vs_H5_Y100""",-13.011736,0,2,"""full_empty""",true,1.7648e-7,"""O00139""",6.7060e-8,686,696,"""R""","""Q""","""A""","""K""","""restricted""","""restricted""","""fully_specific""","""O00139-685_686""","""O00139-696_697""",true,1.7648e-7,"""KIF2A_HUMAN""","""KIF2A""","""Kinesin-like protein KIF2A"""
"""DNYPQSVPR""",-12.991037,6.495518,-80.192237,1.2336e-8,1.7836e-7,8.930737,"""H2d5_Y100_vs_H5_Y100""",-12.993679,0,2,"""full_empty""",true,1.7836e-7,"""O00159""",2.3014e-8,886,894,"""K""","""L""","""D""","""R""","""restricted""","""restricted""","""fully_specific""","""O00159-885_886""","""O00159-894_895""",true,1.7836e-7,"""MYO1C_HUMAN""","""MYO1C""","""Unconventional myosin-Ic"""
